In [0]:
#  Install Great Expectations (Run once)
%pip install great-expectations==0.17.23
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 35.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import great_expectations as ge
from pyspark.sql.functions import col, count as sql_count

In [0]:
# Read Data from Bronze Delta Lake
hotel_booking_bronze_path = "s3://travel-analytics-bronze/delta/bronze/hotel_bookings/"
df_hotel_booking = spark.read.format("delta").load(hotel_booking_bronze_path)

print("=" * 80)
print("HOTEL BOOKING VALIDATION WITH GREAT EXPECTATIONS")
print("=" * 80)
print(f"Total records: {df_hotel_booking.count()}")
print("\n--- Schema ---")
df_hotel_booking.printSchema()

HOTEL BOOKING VALIDATION WITH GREAT EXPECTATIONS
Total records: 10015

--- Schema ---
root
 |-- _airbyte_ab_id: string (nullable = true)
 |-- _airbyte_emitted_at: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- hotel_id: long (nullable = true)
 |-- _ab_cdc_lsn: double (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- booking_date: struct (nullable = true)
 |    |-- member0: date (nullable = true)
 |    |-- member1: string (nullable = true)
 |-- booking_time: string (nullable = true)
 |-- check_in_date: struct (nullable = true)
 |    |-- member0: date (nullable = true)
 |    |-- member1: string (nullable = true)
 |-- check_out_date: struct (nullable = true)
 |    |-- member0: date (nullable = true)
 |    |-- member1: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- _ab_cdc_deleted_at: string (nullable = true)
 |-- _ab_cdc_updated_at: string (nullable = true)
 |-- breakfast_includ

In [0]:
ge_df = ge.from_pandas(df_hotel_booking.toPandas())
print("\nRunning Great Expectations validation...")


Running Great Expectations validation...


In [0]:
# Define and Run Expectations

# Expectation 1: hotel_id NOT NULL
result1 = ge_df.expect_column_values_to_not_be_null(column="hotel_id")
print(f"✓ hotel_id NOT NULL: {result1['success']}")

# Expectation 2: customer_id NOT NULL
result2 = ge_df.expect_column_values_to_not_be_null(column="customer_id")
print(f"✓ customer_id NOT NULL: {result2['success']}")

# Expectation 3: price NOT NULL
result3 = ge_df.expect_column_values_to_not_be_null(column="price")
print(f"✓ price NOT NULL: {result3['success']}")

# Expectation 4: price > 0
result4 = ge_df.expect_column_values_to_be_between(column="price", min_value=0.01, max_value=None)
print(f"✓ price > 0: {result4['success']}")

# Expectation 5: check_in_date NOT NULL
result5 = ge_df.expect_column_values_to_not_be_null(column="check_in_date")
print(f"✓ check_in_date NOT NULL: {result5['success']}")

# Expectation 6: payment_status NOT NULL
result6 = ge_df.expect_column_values_to_not_be_null(column="payment_status")
print(f"✓ payment_status NOT NULL: {result6['success']}")

✓ hotel_id NOT NULL: True
✓ customer_id NOT NULL: True
✓ price NOT NULL: True
✓ price > 0: True
✓ check_in_date NOT NULL: True
✓ payment_status NOT NULL: True


In [0]:
# Apply Validations using PySpark

print("\n--- Applying Validations ---")

# Start with all data
df_valid = df_hotel_booking
df_invalid_list = []

# Rule 1: hotel_id NOT NULL
df_invalid_hotel = df_valid.filter(col("hotel_id").isNull())
if df_invalid_hotel.count() > 0:
    df_invalid_list.append(df_invalid_hotel)
    print(f"  → Found {df_invalid_hotel.count()} rows with hotel_id = NULL")
df_valid = df_valid.filter(col("hotel_id").isNotNull())

# Rule 2: customer_id NOT NULL
df_invalid_customer = df_valid.filter(col("customer_id").isNull())
if df_invalid_customer.count() > 0:
    df_invalid_list.append(df_invalid_customer)
    print(f"  → Found {df_invalid_customer.count()} rows with customer_id = NULL")
df_valid = df_valid.filter(col("customer_id").isNotNull())

# Rule 3: price NOT NULL
df_invalid_price_null = df_valid.filter(col("price").isNull())
if df_invalid_price_null.count() > 0:
    df_invalid_list.append(df_invalid_price_null)
    print(f"  → Found {df_invalid_price_null.count()} rows with price = NULL")
df_valid = df_valid.filter(col("price").isNotNull())

# Rule 4: price > 0
df_invalid_price_negative = df_valid.filter(col("price") <= 0)
if df_invalid_price_negative.count() > 0:
    df_invalid_list.append(df_invalid_price_negative)
    print(f"  → Found {df_invalid_price_negative.count()} rows with price <= 0")
df_valid = df_valid.filter(col("price") > 0)

# Rule 5: check_in_date NOT NULL
df_invalid_checkin = df_valid.filter(col("check_in_date").isNull())
if df_invalid_checkin.count() > 0:
    df_invalid_list.append(df_invalid_checkin)
    print(f"  → Found {df_invalid_checkin.count()} rows with check_in_date = NULL")
df_valid = df_valid.filter(col("check_in_date").isNotNull())

# Rule 6: payment_status NOT NULL
df_invalid_payment = df_valid.filter(col("payment_status").isNull())
if df_invalid_payment.count() > 0:
    df_invalid_list.append(df_invalid_payment)
    print(f"  → Found {df_invalid_payment.count()} rows with payment_status = NULL")
df_valid = df_valid.filter(col("payment_status").isNotNull())


--- Applying Validations ---


In [0]:
# Combine Invalid Records

if df_invalid_list:
    df_invalid = df_invalid_list[0]
    for df_temp in df_invalid_list[1:]:
        df_invalid = df_invalid.union(df_temp)
    df_invalid = df_invalid.distinct()
else:
    df_invalid = spark.createDataFrame([], df_hotel_booking.schema)

valid_count = df_valid.count()
invalid_count = df_invalid.count()
total_count = df_hotel_booking.count()

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)
print(f"✅ Valid records:   {valid_count} ({valid_count/total_count*100:.2f}%)")
print(f"❌ Invalid records: {invalid_count} ({invalid_count/total_count*100:.2f}%)")



VALIDATION RESULTS
✅ Valid records:   10015 (100.00%)
❌ Invalid records: 0 (0.00%)


In [0]:
# Write Invalid Records to Quarantine

if invalid_count > 0:
    quarantine_path = "s3://travel-analytics-bronze/Quarantine/Hotel_Bookings"
    
    df_invalid.write \
        .format("parquet") \
        .mode("append") \
        .save(quarantine_path)
    
    print(f"\n❌ Invalid records sent to Quarantine: {quarantine_path}")
    print("\n--- Sample Invalid Records ---")
    df_invalid.show(10, truncate=False)
else:
    print(f"\n✅ All {valid_count} records passed validation!")

print("\n" + "=" * 80)
print("✅ VALIDATION COMPLETED!")
print("=" * 80)
print(f"Valid records remain in Bronze: {hotel_booking_bronze_path}")
if invalid_count > 0:
    print(f"Invalid records in Quarantine: s3://travel-analytics-bronze/Quarantine/Hotel_Bookings")


✅ All 10015 records passed validation!

✅ VALIDATION COMPLETED!
Valid records remain in Bronze: s3://travel-analytics-bronze/delta/bronze/hotel_bookings/
